# Unsupervised Analysis

This notebook evaluates the structure of the preprocessed lamproite geochemical dataset using unsupervised multivariate methods. Principal component analysis is performed first using the log transformed and Z score standardised dataset.

In [ ]:
# =============================================================================
# Environment setup
# =============================================================================

import os
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.decomposition import PCA

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORT_DIR = ROOT_DIR / "exports"
FIGURE_DIR = ROOT_DIR / "figures"

EXPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 99

print("Environment initialised.")
print(f"Root directory   : {ROOT_DIR}")
print(f"Processed data   : {PROCESSED_DIR}")
print(f"Export directory : {EXPORT_DIR}")
print(f"Figure directory : {FIGURE_DIR}")

## 1. Load Preprocessed Dataset

Load the log transformed and Z score standardised modelling dataset and evaluate the explained variance of the principal components to determine the number of components required to capture the majority of the dataset variance.

In [ ]:
# =============================================================================
# Load preprocessed dataset
# =============================================================================

DATA_FILE = PROCESSED_DIR / "Modelling_Dataset_Log_ZScaled.csv"
ELEMENT_FILE = EXPORT_DIR / "04_retained_trace_elements.csv"

df = pd.read_csv(DATA_FILE, encoding="utf-8")

META = ["Ref", "Country", "Series"]

ELEMENTS = pd.read_csv(
    ELEMENT_FILE,
    encoding="utf-8"
)["Element"].tolist()

X = df[ELEMENTS].copy()

print("Preprocessed dataset loaded.")

print(f"Analyses : {len(df)}")
print(f"Elements : {len(ELEMENTS)}")
print(ELEMENTS)

## 2. Principal Component Analysis

Fit a principal component analysis using all retained trace elements and evaluate the explained and cumulative variance to determine the number of principal components retained for subsequent analyses.

In [ ]:
# =============================================================================
# Principal component analysis
# =============================================================================

pca = PCA(random_state=RANDOM_STATE)
scores = pca.fit_transform(X)

explained_variance = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(ELEMENTS))],
    "ExplainedVariance(%)": np.round(pca.explained_variance_ratio_ * 100, 2),
    "CumulativeVariance(%)": np.round(np.cumsum(pca.explained_variance_ratio_) * 100, 2)
})

display(explained_variance)

In [ ]:
# =============================================================================
# PCA loading plots
# =============================================================================

loadings = pd.DataFrame(pca.components_.T, index=ELEMENTS,
                        columns=[f"PC{i+1}" for i in range(len(ELEMENTS))])
loadings.round(4).to_csv(EXPORT_DIR / "05_PCA_Loadings.csv")

fig, axes = plt.subplots(4, 4, figsize=(20, 15.2))
axes = np.array(axes).flatten()
plot_axes = list(axes[:12]) + [axes[13], axes[14]]

for ax, pc in zip(plot_axes, loadings.columns):
    values = loadings[pc]
    colours = ["firebrick" if abs(v) >= 0.60 else "darkorange" if abs(v) >= 0.30
               else "lightgrey" for v in values]

    ax.bar(ELEMENTS, values, color=colours, edgecolor="black", linewidth=0.5, width=0.5)
    ax.axhline(0, color="black", linewidth=1)

    pc_num = int(pc[2:])
    variance = pca.explained_variance_ratio_[pc_num-1] * 100
    ax.set_title(f"{pc} ({variance:.1f}%)", fontsize=16)
    ax.tick_params(labelsize=14)
    ax.grid(axis="y", alpha=0.3)

axes[12].remove()
axes[15].remove()

fig.supylabel("Loading weight", fontsize=16)

legend_handles = [
    Line2D([0], [0], color="firebrick", lw=6, label="Strong |loading| ≥ 0.60"),
    Line2D([0], [0], color="darkorange", lw=6, label="Moderate 0.30 ≤ |loading| < 0.60"),
    Line2D([0], [0], color="lightgrey", lw=6, label="Weak |loading| < 0.30")
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 0.01), fontsize=12)

plt.tight_layout(rect=[0.02, 0.06, 1, 1])
outpath = FIGURE_DIR / "07_PCA_Loading_Plots.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure : {outpath}")
print(f"Saved table  : {EXPORT_DIR / '05_PCA_Loadings.csv'}")

In [ ]:
# =============================================================================
# PCA score distributions for all principal components
# =============================================================================

scores = pd.DataFrame(pca.transform(X), columns=[f"PC{i+1}" for i in range(len(ELEMENTS))])
scores = pd.concat([df[META].reset_index(drop=True), scores], axis=1)
pcs = [f"PC{i}" for i in range(1, len(ELEMENTS)+1)]

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = np.array(axes).flatten()
plot_axes = list(axes[:12]) + [axes[13], axes[14]]
series_colours = {"Orogenic": "tab:blue", "Anorogenic": "tab:red"}

for ax, pc in zip(plot_axes, pcs):
    bins = np.histogram_bin_edges(scores[pc].dropna(), bins="fd")

    for series, colour in series_colours.items():
        ax.hist(scores.loc[scores["Series"].eq(series), pc], bins=bins, density=True,
                alpha=0.4, color=colour, edgecolor="black", linewidth=0.4)
        scores.loc[scores["Series"].eq(series), pc].plot.kde(
            ax=ax, color=colour, linewidth=2
        )

    i = int(pc[2:])-1
    ax.set_xlabel(f"{pc} ({pca.explained_variance_ratio_[i]*100:.1f}%)", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(alpha=0.3)

axes[12].remove()
axes[15].remove()

legend_handles = [
    Line2D([0], [0], color="tab:blue", lw=6, label="Orogenic"),
    Line2D([0], [0], color="tab:red", lw=6, label="Anorogenic")
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, 0.01), fontsize=11)

plt.tight_layout(rect=[0, 0.04, 1, 1])
outpath = FIGURE_DIR / "08_PC_Distributions.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {outpath}")

In [ ]:
# =============================================================================
# Orogenic–Anorogenic separation along individual principal components
# =============================================================================

rows = []

for pc in pcs:
    oro = scores.loc[scores["Series"].eq("Orogenic"), pc].dropna()
    ano = scores.loc[scores["Series"].eq("Anorogenic"), pc].dropna()

    oro_median, ano_median = oro.median(), ano.median()
    oro_mad = np.median(np.abs(oro-oro_median))
    ano_mad = np.median(np.abs(ano-ano_median))
    separation = abs(oro_median-ano_median) / np.sqrt(oro_mad**2+ano_mad**2)

    i = int(pc[2:])-1
    rows.append({
        "PC": pc,
        "ExplainedVariance(%)": pca.explained_variance_ratio_[i]*100,
        "Orogenic_Median": oro_median,
        "Orogenic_MAD": oro_mad,
        "Anorogenic_Median": ano_median,
        "Anorogenic_MAD": ano_mad,
        "MAD_Normalised_Separation": separation
    })

pc_separation = pd.DataFrame(rows)
pc_separation.iloc[:, 1:] = pc_separation.iloc[:, 1:].round(3)
pc_separation = pc_separation.sort_values("MAD_Normalised_Separation", ascending=False).reset_index(drop=True)

display(pc_separation)

outpath = EXPORT_DIR / "06_PC_Separation.csv"
pc_separation.to_csv(outpath, index=False)
print(f"Saved: {outpath}")

In [ ]:
# =============================================================================
# PCA scatter plot with Orogenic-only, Anorogenic-only and overlap fields
# =============================================================================

from scipy.stats import gaussian_kde
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

pcx, pcy = "PC1", "PC2"
oro = scores[scores["Series"].eq("Orogenic")]
ano = scores[scores["Series"].eq("Anorogenic")]

def hdr_mask(x, y, xx, yy, prob=0.95):
    kde = gaussian_kde(np.vstack([x, y]))
    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    zsort = np.sort(zz.ravel())[::-1]
    csum = np.cumsum(zsort) / zsort.sum()
    level = zsort[np.searchsorted(csum, prob)]
    return zz >= level, kde, level

xmin, xmax = scores[pcx].min()-0.5, scores[pcx].max()+0.5
ymin, ymax = scores[pcy].min()-0.5, scores[pcy].max()+0.5
xx, yy = np.meshgrid(np.linspace(xmin, xmax, 400), np.linspace(ymin, ymax, 400))

oro_mask, oro_kde, oro_level = hdr_mask(oro[pcx].values, oro[pcy].values, xx, yy)
ano_mask, ano_kde, ano_level = hdr_mask(ano[pcx].values, ano[pcy].values, xx, yy)

zone = np.zeros_like(xx, dtype=int)
zone[oro_mask & ~ano_mask] = 1
zone[ano_mask & ~oro_mask] = 2
zone[oro_mask & ano_mask] = 3

all_xy = np.vstack([scores[pcx], scores[pcy]])
oro_points = oro_kde(all_xy) >= oro_level
ano_points = ano_kde(all_xy) >= ano_level

n_oro_only = np.sum(oro_points & ~ano_points)
n_ano_only = np.sum(ano_points & ~oro_points)
n_overlap = np.sum(oro_points & ano_points)

cmap = ListedColormap(["white", "tab:blue", "tab:red", "purple"])

fig, ax = plt.subplots(figsize=(10, 8))
ax.pcolormesh(xx, yy, zone, cmap=cmap, shading="auto", alpha=0.18)
ax.contour(xx, yy, oro_mask.astype(int), levels=[0.5], colors=["tab:blue"], linewidths=2)
ax.contour(xx, yy, ano_mask.astype(int), levels=[0.5], colors=["tab:red"], linewidths=2)

ax.scatter(oro[pcx], oro[pcy], s=45, color="tab:blue", alpha=0.55,
           edgecolor="black", linewidth=0.4)
ax.scatter(ano[pcx], ano[pcy], s=45, color="tab:red", alpha=0.55,
           edgecolor="black", linewidth=0.4)

ax.axhline(0, color="grey", lw=1)
ax.axvline(0, color="grey", lw=1)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=14)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=14)
ax.tick_params(labelsize=12)
ax.grid(alpha=0.3)

legend_handles = [
    Patch(facecolor="tab:blue", edgecolor="black", alpha=0.3,
          label=f"Orogenic only (N={n_oro_only})"),
    Patch(facecolor="tab:red", edgecolor="black", alpha=0.3,
          label=f"Anorogenic only (N={n_ano_only})"),
    Patch(facecolor="purple", edgecolor="black", alpha=0.3,
          label=f"Overlap (N={n_overlap})")
]
ax.legend(handles=legend_handles, frameon=True, edgecolor="black", fontsize=12,
          loc="upper right")

plt.tight_layout()
outpath = FIGURE_DIR / "09_PC1_PC2_Score_Fields.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {outpath}")